## Inference under FHE for the MNIST Dataset using helayers

In this demo, we'll deal with a classification problem for the MNIST dataset [1], trying to correctly classify a batch of samples using a neural network model that will be created and trained using the Keras library (with architecture similar to reference [2]).
First, we'll build a plain neural network for the MNIST model. Then, we'll encrypt the trained network and run inference over it using FHE.

In [1]:
import os

##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.random.set_seed(seed_value)
from tensorflow.keras import backend as K

from tensorflow.keras import utils, losses
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
import h5py

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)

batch_size = 500
epochs = 10
print("Misc. initializations")

2024-12-29 11:20:04.866833: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Misc. initializations


### Load and Preprocess the MNIST Dataset. 

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

x_train /= 255
x_test /= 255
print('data ready')



data ready


In [3]:
# Create validation data
testSize=16
x_val = x_test[:-testSize]
x_test = x_test[-testSize:]
y_val = y_test[:-testSize]
y_test = y_test[-testSize:]
print('Validation and test data ready')

# Convert class vector to binary class matrices
num_classes = 10
y_train = utils.to_categorical(y_train, num_classes)
y_test = utils.to_categorical(y_test, num_classes)
y_val = utils.to_categorical(y_val, num_classes)

input_shape = x_train[0].shape
print(f'input shape: {input_shape}')

Validation and test data ready
input shape: (28, 28, 1)


### Save Dataset

In [4]:
def save_data_set(x, y, data_type, s=''):
    fname=os.path.join(PATH, f'x_{data_type}{s}.h5')
    print("Saving x_{} of shape {} in {}".format(data_type, x.shape,fname))
    xf = h5py.File(fname, 'w')
    xf.create_dataset('x_{}'.format(data_type), data=x)
    xf.close()

    yf = h5py.File(os.path.join(PATH, f'y_{data_type}{s}.h5'), 'w')
    yf.create_dataset(f'y_{data_type}', data=y)
    yf.close()

save_data_set(x_test, y_test, data_type='test')
save_data_set(x_train, y_train, data_type='train')
save_data_set(x_val, y_val, data_type='val')

Saving x_test of shape (16, 28, 28, 1) in data/net_mnist/x_test.h5
Saving x_train of shape (60000, 28, 28, 1) in data/net_mnist/x_train.h5
Saving x_val of shape (9984, 28, 28, 1) in data/net_mnist/x_val.h5


### Build a Plain Neural Network for the MNIST Model

In [5]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D
from data_gen.activations import SquareActivation
model = Sequential()
# 第一个卷积层
model.add(Conv2D(20, kernel_size=5, input_shape=(28, 28, 1)))  # 假设输入图像大小为28x28，通道数为1
# 第一个平均池化层
model.add(AvgPool2D(pool_size=2))

# 展平层，将多维输入展平为一维
model.add(Flatten())
# 第一个全连接层
model.add(Dense(500))
# 批量归一化层
model.add(BatchNormalization())
# ReLU 激活层
model.add(SquareActivation())
# 第二个全连接层（输出层）
model.add(Dense(10))  # 10 类的输出

# 输出模型架构

model.summary()

2024-12-29 11:20:14.695097: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-12-29 11:20:14.853735: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-12-29 11:20:14.857037: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-12-29 11:20:14.860996: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the approp

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 24, 24, 20)        520       
                                                                 
 average_pooling2d (AverageP  (None, 12, 12, 20)       0         
 ooling2D)                                                       
                                                                 
 flatten (Flatten)           (None, 2880)              0         
                                                                 
 dense (Dense)               (None, 500)               1440500   
                                                                 
 batch_normalization (BatchN  (None, 500)              2000      
 ormalization)                                                   
                                                                 
 square_activation (SquareAc  (None, 500)              0

### Train the Neural Network

In [6]:
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)

model.compile(loss=sum_squared_error,
                  optimizer='Adam',
                  metrics=['accuracy'])

model.fit(x_train, y_train,
              batch_size=batch_size,
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True,
              )

score = model.evaluate(x_test, y_test, verbose=0)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/10


2024-12-29 11:20:24.428401: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8500


120/120 - 9s - loss: 0.8783 - accuracy: 0.8538 - val_loss: 0.5973 - val_accuracy: 0.8914 - 9s/epoch - 72ms/step
Epoch 2/10
120/120 - 0s - loss: 0.2089 - accuracy: 0.9500 - val_loss: 0.3405 - val_accuracy: 0.9470 - 500ms/epoch - 4ms/step
Epoch 3/10
120/120 - 0s - loss: 0.1911 - accuracy: 0.9563 - val_loss: 0.2543 - val_accuracy: 0.9503 - 498ms/epoch - 4ms/step
Epoch 4/10
120/120 - 1s - loss: 0.1824 - accuracy: 0.9610 - val_loss: 0.1933 - val_accuracy: 0.9619 - 502ms/epoch - 4ms/step
Epoch 5/10
120/120 - 0s - loss: 0.1746 - accuracy: 0.9639 - val_loss: 0.1908 - val_accuracy: 0.9630 - 498ms/epoch - 4ms/step
Epoch 6/10
120/120 - 0s - loss: 0.1677 - accuracy: 0.9650 - val_loss: 0.1912 - val_accuracy: 0.9601 - 496ms/epoch - 4ms/step
Epoch 7/10
120/120 - 1s - loss: 0.1630 - accuracy: 0.9671 - val_loss: 0.1733 - val_accuracy: 0.9639 - 503ms/epoch - 4ms/step
Epoch 8/10
120/120 - 0s - loss: 0.1610 - accuracy: 0.9684 - val_loss: 0.1756 - val_accuracy: 0.9646 - 497ms/epoch - 4ms/step
Epoch 9/10
12

### Report the Confusion Matrix of the Trained Model

In [7]:
from sklearn import metrics

y_pred_vals = model.predict(x_test)
y_pred = np.argmax(y_pred_vals, axis=1)
y_test = np.argmax(y_test, axis=1)
cm = metrics.confusion_matrix(y_test, y_pred)
print(cm)

1/1 [==============================] - 0s 89ms/step
[[1 0 0 0 0 0 0 0 0 0]
 [0 2 0 0 0 0 0 0 0 0]
 [0 0 2 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 1]]


### Serialize Model and Weights

In [8]:
model_json = model.to_json()
with open(os.path.join(PATH, 'model.json'), "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
model.save_weights(os.path.join(PATH, 'model.h5'))
print("Saved model to disk")

Saved model to disk


We are all done training the plain network. Next we will encrypt the network and run inference over it using FHE. Let's start with some initializations.

In [9]:
import pyhelayers
import utils

utils.verify_memory()

print('Misc. initializations')

Misc. initializations


The following is a general outline of the next steps.

We encode and encrypt a neural network model, using the files that were created and saved before. An automated optimizer, that occurs during the call to encode_encrypt, will examine our network and will determine various configuration details that will allow running inference under encryption efficiently.

Next, we will demonstrate how we can encrypt data, run inference on our encrypted network, and compare the results against the expected labels.
Now let's dive in . . .

In [10]:
he_run_req = pyhelayers.HeRunRequirements()
he_run_req.set_he_context_options([pyhelayers.DefaultContext()])
he_run_req.optimize_for_batch_size(16)

nn = pyhelayers.NeuralNet()
nn.encode_encrypt([os.path.join(PATH, "model.json"), os.path.join(PATH, "model.h5")], he_run_req)

The encode_encrypt method also initialized an HeContext object containing the keys. We obtain it now from the model so we can encrypt the input data.

In [11]:
context = nn.get_created_he_context()

We will now load real samples of the MNIST dataset to classify. We will load the samples and the corresponding true labels from HDF5 files. We will also extract the first batch of samples and labels.

In [12]:
with h5py.File(os.path.join(PATH, "x_test.h5")) as f:
    x_test = np.array(f["x_test"])
with h5py.File(os.path.join(PATH, "y_test.h5")) as f:
    y_test = np.array(f["y_test"])
    
# plain_samples, labels = utils.extract_batch(x_test, y_test, batch_size, 0)

plain_samples, labels = utils.extract_batch(x_test, y_test, 100, 0)
print('Batch of size',batch_size,'loaded')

Batch of size 500 loaded


To populate the input, we need to encode and then encrypt the values of the plain input under HE.

In [13]:
model_io_encoder = pyhelayers.ModelIoEncoder(nn)
samples = pyhelayers.EncryptedData(context)
model_io_encoder.encode_encrypt(samples, [plain_samples])
print('Test data encrypted')

Test data encrypted


We now go ahead with the inference itself. We run the encrypted input through the encrypted NN to obtain encrypted results. This computation does not use the secret key and acts on completely encrypted values. Running the inference is done using the "predict" method of the NN, that receives the destination 3D structure to put the result of the computation in, and the input for the inference.

In [ ]:
utils.start_timer()

predictions = pyhelayers.EncryptedData(context)
nn.predict(predictions, samples)

duration=utils.end_timer('predict')
utils.report_duration('predict per sample',duration/batch_size)

In order to assess the results of the computation, we first need to decrypt them. This is done by an IO processor that has the secret key and is capable of decrypting the ciphertext and decoding it into plaintext version of the HE computation result.

In [ ]:
plain_predictions = model_io_encoder.decrypt_decode_output(predictions)
print('predictions',plain_predictions)

Now we compare the results against the expected labels and compute the confusion matrix and the accuracy.

In [ ]:
utils.assess_results(labels, plain_predictions)

<br>

References:

<sub><sup> 1.	LeCun, Yann and Cortes, Corinna. "MNIST handwritten digit database." (2010): </sup></sub>

<sub><sup> 2.	Gilad-Bachrach, R., Dowlin, N., Laine, K., Lauter, K., Naehrig, M. &amp; Wernsing, J.. (2016). CryptoNets: Applying Neural Networks to Encrypted Data with High Throughput and Accuracy. Proceedings of The 33rd International Conference on Machine Learning, in Proceedings of Machine Learning Research 48:201-210 Available from https://proceedings.mlr.press/v48/gilad-bachrach16.html.
</sup></sub>
